# HP Meta-Model Training (Visual)

This notebook trains the HP meta-model **per task type** and visualizes each step.

**Overview:**
1. Load `hp_tuning_db` CSV
2. Split by `task_type` (e.g. classification / regression)
3. For each task type: data prep → Scorer → Direct Predictor → Ranker → Archetypes → save to subdir
4. Aggregate CV metrics and output directory

**Environment:** Install `lightgbm`, `scikit-learn`, `pandas` in `../venv` (or the current kernel).

**Next:** After training, run `evaluate_hp_meta_model.ipynb` for offline evaluation (same HP DB and model dir).

## 1. Configuration and dependencies

In [1]:
import os
import json
import numpy as np
import pandas as pd
import sys

# Walk up to find directory containing train_hp_meta_model.py and add to sys.path
_here = os.path.abspath(os.getcwd())
_root = _here
while _root and _root != os.path.dirname(_root):
    if os.path.isfile(os.path.join(_root, "train_hp_meta_model.py")):
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break
    _root = os.path.dirname(_root)
else:
    if _here not in sys.path:
        sys.path.insert(0, _here)

# Training config (edit as needed)
HP_DB_PATH = "hp_tuning_db_refined.csv"   # HP tuning DB CSV
OUTPUT_DIR = "./hp_meta_model"             # Model output directory
TOP_K = 3                                 # Top-K configs per dataset for Direct Predictor training
N_CLUSTERS = 2                            # Number of archetype clusters (e.g. 2 for Spectral from 2_compare)
CLUSTER_METHOD = "spectral"               # "kmeans" or "spectral" (from 2_compare_archetype_clustering)
PER_TASK_TYPE = True                      # True = train per task type; False = single model (legacy)
HOLDOUT_RATIO = 0.2                       # Fraction of datasets to hold out for evaluation (0 = no hold-out). Enables fair eval in 4_evaluate.
HOLDOUT_SEED = 42                         # Random seed for hold-out split

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"HP DB: {HP_DB_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Per-task training: {PER_TASK_TYPE}")
print(f"Hold-out ratio: {HOLDOUT_RATIO} (0 = no hold-out)")
print(f"Archetype: {CLUSTER_METHOD}, n_clusters={N_CLUSTERS}")

HP DB: hp_tuning_db_refined.csv
Output: ./hp_meta_model
Per-task training: True
Hold-out ratio: 0.2 (0 = no hold-out)


In [2]:
# Import from training module (logic in train_hp_meta_model.py)
from train_hp_meta_model import (
    train_hp_meta_model,
    print_hp_analysis,
    HPDataPreparator,
    HPScorerTrainer,
    DirectHPPredictor,
    HPRankerTrainer,
    compute_archetypes,
    HPEnsemblePredictor,
    _train_single_task,
    MIN_SAMPLES_PER_TASK,
)
print("Import OK.")

Import OK.


## 2. Load data and task type distribution

In [3]:
df = pd.read_csv(HP_DB_PATH)
print(f"Total rows: {len(df)}")
print(f"Datasets: {df['dataset_name'].nunique() if 'dataset_name' in df.columns else '?'}")

if 'task_type' in df.columns:
    task_counts = df['task_type'].value_counts()
    display(task_counts.to_frame(name='count'))
else:
    print("No 'task_type' column; will train as single task 'all'.")

Total rows: 135295
Datasets: 1749


,count
task_type,
classification,105097
regression,30198


## 3. HP analysis (optional)

View tuning gain, score distribution, and HP–performance correlations.

In [4]:
print_hp_analysis(df)


HP SENSITIVITY ANALYSIS

  Tuning vs Default (1749 datasets):
    Average improvement:  +1893.64749
    Best-case gain:       +5556280.83078
    % datasets improved:  40.9%

  Score Range within Datasets:
    Average range:  12513.43327
    Median range:   0.04246
    Max range:      12121772.13558

  HP-Performance Correlations (Spearman):
    hp_colsample_bytree             r=-0.1020  ***
    hp_n_estimators                 r=+0.0403  ***
    hp_max_bin                      r=-0.0316  ***
    hp_reg_lambda                   r=-0.0235  ***
    hp_min_child_samples            r=+0.0225  ***
    hp_learning_rate                r=-0.0225  ***
    hp_max_depth                    r=+0.0209  ***
    hp_subsample                    r=+0.0162  ***
    hp_num_leaves                   r=+0.0077  **
    hp_reg_alpha                    r=-0.0053  

  Optimal HP Distributions (best config per dataset):
    hp_num_leaves                   median=  105.0000  IQR=[43.0000, 180.0000]  range=[4.0000, 

## 4. Per-task training (visual)

For each `task_type`: **data prep → Scorer → Direct Predictor → Ranker → Archetypes → save**. Task types with fewer than `MIN_SAMPLES_PER_TASK` samples are skipped.

In [5]:
# Hold-out split (by dataset) so evaluation can run on unseen datasets and avoid leakage
holdout_datasets = []
if HOLDOUT_RATIO > 0 and 'dataset_name' in df.columns:
    all_ds = df['dataset_name'].unique().tolist()
    n = len(all_ds)
    if n >= 2:
        n_holdout = max(1, min(n - 1, int(round(n * HOLDOUT_RATIO))))
        rng = np.random.RandomState(HOLDOUT_SEED)
        perm = rng.permutation(n)
        holdout_datasets = [all_ds[i] for i in perm[-n_holdout:]]
        holdout_set = set(holdout_datasets)
        df = df[~df['dataset_name'].isin(holdout_set)].copy()
        print(f"Hold-out: {n_holdout} datasets excluded for evaluation; training on {n - n_holdout} datasets.")

if not PER_TASK_TYPE or 'task_type' not in df.columns:
    task_types = ['all']
    df_per_task = {'all': df}
else:
    task_types = df['task_type'].dropna().unique().tolist()
    task_types = [str(tt).strip() for tt in task_types if str(tt).strip()]
    df_per_task = {tt: df[df['task_type'] == tt].copy().reset_index(drop=True) for tt in task_types}

if task_types == ['all']:
    train_hp_meta_model(HP_DB_PATH, OUTPUT_DIR, top_k=TOP_K, n_clusters=N_CLUSTERS, per_task_type=False,
                        holdout_ratio=HOLDOUT_RATIO, holdout_seed=HOLDOUT_SEED, cluster_method=CLUSTER_METHOD)
else:
    results = []
    for task_type in sorted(task_types):
        df_tt = df_per_task[task_type]
        if len(df_tt) < MIN_SAMPLES_PER_TASK:
            print(f"Skipping task_type '{task_type}': {len(df_tt)} samples < {MIN_SAMPLES_PER_TASK}")
            continue
        out_subdir = os.path.join(OUTPUT_DIR, task_type)
        r = _train_single_task(df_tt, task_type, out_subdir, top_k=TOP_K, n_clusters=N_CLUSTERS, cluster_method=CLUSTER_METHOD)
        results.append(r)
    cv_summary = {}
    for r in results:
        tt = r['task_type']
        c = r['cv_scores']
        cv_summary[tt] = {
            'n_samples': r['n_samples'],
            'n_datasets': r['n_datasets'],
            'scorer_r2': (c.get('scorer') or {}).get('r2_mean'),
            'ranker_ndcg': c.get('ranker'),
        }
    manifest = {
        'per_task': True,
        'task_types': [r['task_type'] for r in results],
        'cv_summary': cv_summary,
    }
    with open(os.path.join(OUTPUT_DIR, 'hp_manifest.json'), 'w') as f:
        json.dump(manifest, f, indent=2, default=str)
    if holdout_datasets:
        holdout_meta = {
            'holdout_ratio': HOLDOUT_RATIO,
            'seed': HOLDOUT_SEED,
            'holdout_datasets': sorted(holdout_datasets),
            'n_holdout': len(holdout_datasets),
            'n_train': df['dataset_name'].nunique(),
        }
        with open(os.path.join(OUTPUT_DIR, 'holdout_datasets.json'), 'w') as f:
            json.dump(holdout_meta, f, indent=2)
        print(f"  Hold-out list saved: {OUTPUT_DIR}/holdout_datasets.json ({len(holdout_datasets)} datasets)")
    print(f"\n>> Written to {OUTPUT_DIR}/hp_manifest.json")


Hold-out: 350 datasets excluded for evaluation; training on 1399 datasets.

TASK TYPE: classification  (85095 rows, 1114 datasets)
  After dropping NaN primary_score: 85095 rows (was 85095)
  Task 'classification': 85095 rows, raw score mean=0.80684, std=0.16733
  Top-3 configs per dataset: 5251 rows from 1114 datasets (avg 4.7/dataset)
  Scorer features: 54, Predictor features: 34

Training HP Config Scorer (regressor)...
  Samples: 85095, Features: 54
  Using GroupKFold (5 folds, 1114 datasets)
  CV MAE:  0.21766 +/- 0.01919
  CV RMSE: 0.36311 +/- 0.03421
  CV R2:   0.8670 +/- 0.0251
  Final model: n_estimators=750 (avg CV best_iteration, range 552-986)

  Top 15 features:
    landmarking_score                          6.6% ###
    class_imbalance_ratio                      6.4% ###
    linearity_gap                              5.5% ##
    std_feature_importance                     4.0% ##
    max_target_corr                            3.8% #
    ix_data_per_leaf                    

## 5. Training results summary

In [6]:
manifest_path = os.path.join(OUTPUT_DIR, 'hp_manifest.json')
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    print("Manifest:", json.dumps(manifest, indent=2, default=str))

    if manifest.get('cv_summary'):
        summary = pd.DataFrame(manifest['cv_summary']).T
        display(summary)
else:
    print("hp_manifest.json not found; run the training cell above first.")

Manifest: {
  "per_task": true,
  "task_types": [
    "classification",
    "regression"
  ],
  "cv_summary": {
    "classification": {
      "n_samples": 85095,
      "n_datasets": 1114,
      "scorer_r2": 0.8669919841665514,
      "ranker_ndcg": {
        "ndcg@3_mean": 0.6483690546195315,
        "ndcg@3_std": 0.01673211565171487,
        "ndcg@5_mean": 0.6408485372755937,
        "ndcg@5_std": 0.014918453671550631,
        "ndcg@10_mean": 0.6304654825451771,
        "ndcg@10_std": 0.012443016819228287
      }
    },
    "regression": {
      "n_samples": 22918,
      "n_datasets": 286,
      "scorer_r2": -632.7225072257996,
      "ranker_ndcg": {
        "ndcg@3_mean": 0.6397588653319384,
        "ndcg@3_std": 0.013053007640175595,
        "ndcg@5_mean": 0.6181768863755372,
        "ndcg@5_std": 0.016486018289966826,
        "ndcg@10_mean": 0.6027992013693397,
        "ndcg@10_std": 0.017295246878269867
      }
    }
  }
}


,n_samples,n_datasets,scorer_r2,ranker_ndcg
classification,85095,1114,0.866992,"{'ndcg@3_mean': 0.6483690546195315, 'ndcg@3_st..."
regression,22918,286,-632.722507,"{'ndcg@3_mean': 0.6397588653319384, 'ndcg@3_st..."


In [7]:
# Artifacts per task-type directory
for name in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, name)
    if os.path.isdir(path):
        files = sorted(os.listdir(path))
        print(f"{name}/  ({len(files)} files)")
        for f in files:
            print(f"  - {f}")
    else:
        print(f"{name}")

classification/  (9 files)
  - hp_archetypes.json
  - hp_archetypes.pkl
  - hp_cv_scores.json
  - hp_ensemble_meta.json
  - hp_metadata.json
  - hp_predictor.pkl
  - hp_preparator.pkl
  - hp_ranker.pkl
  - hp_scorer.pkl
holdout_datasets.json
hp_archetypes.json
hp_archetypes.pkl
hp_cv_scores.json
hp_ensemble_meta.json
hp_manifest.json
hp_metadata.json
hp_predictor.pkl
hp_preparator.pkl
hp_ranker.pkl
hp_scorer.pkl
regression/  (9 files)
  - hp_archetypes.json
  - hp_archetypes.pkl
  - hp_cv_scores.json
  - hp_ensemble_meta.json
  - hp_metadata.json
  - hp_predictor.pkl
  - hp_preparator.pkl
  - hp_ranker.pkl
  - hp_scorer.pkl


## 6. Inference example (load by task type)

After per-task training, specify `task_type` when loading the ensemble.

In [8]:
# Example: load ensemble by task type (read manifest first, then load)
# If you get "unexpected keyword argument 'task_type'", re-run the import cell above, then this cell
manifest_path = os.path.join(OUTPUT_DIR, 'hp_manifest.json')
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    if manifest.get('per_task') and manifest.get('task_types'):
        task_type_example = manifest['task_types'][0]
        ensemble = HPEnsemblePredictor.load(OUTPUT_DIR, task_type=task_type_example)
        print(f"Loaded ensemble for task_type='{task_type_example}'.")
        print("Inference: result = ensemble.predict(pd.DataFrame([ds_meta]))")
    else:
        ensemble = HPEnsemblePredictor.load(OUTPUT_DIR)
        print("Loaded single-model ensemble.")
else:
    ensemble = HPEnsemblePredictor.load(OUTPUT_DIR)
    print("Loaded single-model ensemble.")

Loaded ensemble for task_type='classification'.
Inference: result = ensemble.predict(pd.DataFrame([ds_meta]))
